## NASA POWER Climate Data Collection

This notebook explores historical climate data from NASA POWER (Prediction Of Worldwide Energy Resources) to investigate potential relationships between environmental conditions and agricultural productivity.

We will focus on two climate variables: average air temperature and precipitation. We will begin by testing the NASA POWER Application Programming Interface (API) using a single geographic location before expanding data collection to counties in Iowa and Illinois represented in our USDA crop yield dataset.

Our objective is to construct a **county-year climate dataset covering 2015–2025** that can be integrated with the USDA agricultural yield observations collected in Notebook 1.

To support reproducibility and future analysis, we will use reusable data collection functions, local API caching, consistent county identifiers, and validation checks. The resulting processed dataset will be saved for use in Notebook 3, where we will investigate relationships between climate conditions and crop yields.

In [2]:
from pathlib import Path
from requests.exceptions import RequestException

import pandas as pd
import requests
import hashlib
import json
import time

In [3]:
# NASA POWER data dictionary
nasa_data_dictionary = pd.DataFrame([
    {
        "Variable": "T2M",
        "Description": "Daily average air temperature at 2 meters",
        "Unit": "°C",
        "Source": "NASA POWER",
    },
    {
        "Variable": "PRECTOTCORR",
        "Description": "Corrected daily precipitation",
        "Unit": "mm/day",
        "Source": "NASA POWER",
    },
])

nasa_data_dictionary


# Create documentation directory
DOCS_DIR = Path("data/dictionaries")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

# Export dictionary
nasa_data_dictionary.to_csv(
    DOCS_DIR / "NASA_POWER_Data_Dictionary.csv",
    index=False
)

In [4]:
# Project directories
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

# Load cleaned USDA data
USDA_DATA_PATH = PROCESSED_DIR / "USDA_Crop_Yields_Cleaned.csv"

clean_df = pd.read_csv(USDA_DATA_PATH)

print(f"Loaded {len(clean_df):,} USDA observations.")
print(f"Source: {USDA_DATA_PATH}")

Loaded 3,821 USDA observations.
Source: c:\Users\Jenny\OneDrive\ASU\DAT301\agricultural-climate-analysis\data\processed\USDA_Crop_Yields_Cleaned.csv


## Reusable Functions

The following functions support reproducible climate data collection from NASA POWER. They are designed to retrieve observations for different geographic coordinates and time periods without duplicating API request logic.

We will also implement local caching to avoid unnecessary API requests when rerunning the notebook.

In [5]:
def request_with_retry(url, params, max_attempts=3, delay=2):
    """
    Make an HTTP GET request with retries for temporary failures.

    Parameters
    ----------
    url : str
        API endpoint.
    params : dict
        Request query parameters.
    max_attempts : int
        Maximum number of request attempts.
    delay : int or float
        Initial delay between retries in seconds.

    Returns
    -------
    requests.Response
        Successful HTTP response.
    """

    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(
                url,
                params=params,
                timeout=60
            )

            response.raise_for_status()
            return response

        except RequestException as error:
            if attempt == max_attempts:
                raise

            wait_time = delay * (2 ** (attempt - 1))

            print(
                f"Request failed (attempt {attempt}/{max_attempts}): "
                f"{error}"
            )
            print(f"Retrying in {wait_time} seconds...")

            time.sleep(wait_time)

In [6]:
def fetch_nasa_climate(latitude, longitude, start_year, end_year):
    """
    Retrieve daily NASA POWER climate data for a location and date range.

    Uses a local JSON cache to avoid repeated API requests.
    """

    params = {
        "parameters": ",".join(NASA_CONFIG["parameters"]),
        "community": NASA_CONFIG["community"],
        "latitude": latitude,
        "longitude": longitude,
        "start": f"{start_year}0101",
        "end": f"{end_year}1231",
        "format": NASA_CONFIG["format"],
    }

    # Generate a unique cache key from the request configuration
    cache_key = hashlib.sha256(
        json.dumps(
            {
                "url": NASA_API_URL,
                "params": params,
            },
            sort_keys=True
        ).encode("utf-8")
    ).hexdigest()[:16]

    cache_path = NASA_CACHE_DIR / f"nasa_power_{cache_key}.json"

    # Load previously downloaded data
    if cache_path.exists():
        print(f"Loading cached NASA POWER data: {cache_path.name}")

        with cache_path.open("r", encoding="utf-8") as file:
            return json.load(file)

    # Request data if no cache exists
    print(f"Requesting NASA POWER data: {latitude}, {longitude}")

    response = request_with_retry(
        url=NASA_API_URL,
        params=params
    )

    data = response.json()

    # Save successful response to cache
    with cache_path.open("w", encoding="utf-8") as file:
        json.dump(data, file)

    return data

In [7]:
def process_nasa_climate(data):
    """
    Convert a NASA POWER API response into a cleaned daily DataFrame.

    Returns
    -------
    pd.DataFrame
        Daily temperature and precipitation observations.
    """

    # Extract climate observations
    climate_data = data["properties"]["parameter"]

    # Convert nested dictionaries into a DataFrame
    df = pd.DataFrame(climate_data)

    # Convert dates into datetime objects
    df.index = pd.to_datetime(
        df.index,
        format="%Y%m%d"
    )

    df = df.reset_index(names="DATE")

    # Rename columns for readability
    df = df.rename(columns={
        "T2M": "TEMPERATURE_C",
        "PRECTOTCORR": "PRECIPITATION_MM"
    })

    # Convert climate measurements to numeric values
    climate_columns = [
        "TEMPERATURE_C",
        "PRECIPITATION_MM"
    ]

    df[climate_columns] = df[climate_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    # Replace NASA POWER missing-value sentinel
    missing_value = data.get("header", {}).get("fill_value", -999)
    df[climate_columns] = df[climate_columns].replace(
        [missing_value, -999],
        float("nan")
    )

    return df

In [8]:
def validate_daily_climate(daily_df, start_year, end_year):
    """
    Validate the completeness of daily NASA POWER climate observations.

    Raises
    ------
    ValueError
        If dates are missing, duplicated, or climate measurements
        contain missing values.
    """

    expected_dates = pd.date_range(
        start=f"{start_year}-01-01",
        end=f"{end_year}-12-31",
        freq="D"
    )

    actual_dates = pd.DatetimeIndex(daily_df["DATE"])

    missing_dates = expected_dates.difference(actual_dates)
    unexpected_dates = actual_dates.difference(expected_dates)
    duplicate_dates = actual_dates.duplicated().sum()

    climate_columns = [
        "TEMPERATURE_C",
        "PRECIPITATION_MM"
    ]

    missing_measurements = daily_df[climate_columns].isna().sum()

    if (
        len(missing_dates) > 0
        or len(unexpected_dates) > 0
        or duplicate_dates > 0
        or missing_measurements.sum() > 0
    ):
        raise ValueError(
            "Climate data validation failed:\n"
            f"Missing dates: {len(missing_dates)}\n"
            f"Unexpected dates: {len(unexpected_dates)}\n"
            f"Duplicate dates: {duplicate_dates}\n"
            f"Missing measurements:\n{missing_measurements}"
        )

    print(
        f"Validation passed: {len(actual_dates):,} daily observations "
        f"({start_year}–{end_year})."
    )

    return True

In [9]:
def aggregate_annual_climate(climate_df):
    """
    Aggregate daily NASA POWER observations into annual climate statistics.

    Returns
    -------
    pd.DataFrame
        Annual average temperature and total precipitation.
    """

    df = climate_df.copy()

    # Extract year from each observation
    df["YEAR"] = df["DATE"].dt.year

    # Calculate annual climate statistics
    annual_df = (
        df
        .groupby("YEAR")
        .agg(
            AVG_TEMPERATURE_C=("TEMPERATURE_C", "mean"),
            TOTAL_PRECIPITATION_MM=("PRECIPITATION_MM", "sum")
        )
        .reset_index()
    )

    return annual_df

In [10]:
def aggregate_growing_season_climate(
    climate_df,
    start_month=5,
    end_month=9
):
    """
    Aggregate daily climate observations over a growing season.

    Parameters
    ----------
    climate_df : pd.DataFrame
        Daily climate observations containing DATE,
        TEMPERATURE_C, and PRECIPITATION_MM.
    start_month : int
        First month of the growing season (inclusive).
    end_month : int
        Last month of the growing season (inclusive).

    Returns
    -------
    pd.DataFrame
        Growing-season temperature and precipitation by year.
    """

    growing_df = climate_df.copy()

    growing_df["YEAR"] = growing_df["DATE"].dt.year
    growing_df["MONTH"] = growing_df["DATE"].dt.month

    growing_df = growing_df[
        growing_df["MONTH"].between(start_month, end_month)
    ]

    seasonal_df = (
        growing_df.groupby("YEAR")
        .agg(
            GROWING_SEASON_AVG_TEMPERATURE_C=(
                "TEMPERATURE_C", "mean"
            ),
            GROWING_SEASON_TOTAL_PRECIPITATION_MM=(
                "PRECIPITATION_MM", "sum"
            )
        )
        .reset_index()
    )

    return seasonal_df

In [11]:
def download_census_gazetteer(url, cache_path):
    """
    Download and cache a Census Gazetteer geographic reference file.

    Parameters
    ----------
    url : str
        URL of the Census Gazetteer ZIP file.
    cache_path : Path
        Local destination for the cached file.

    Returns
    -------
    Path
        Path to the downloaded or cached ZIP file.
    """

    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    # Reuse the existing file if available
    if cache_path.exists() and cache_path.stat().st_size > 0:
        print(f"Loading cached Census Gazetteer: {cache_path.name}")
        return cache_path

    # Download the geographic reference file
    print("Downloading Census Gazetteer reference data...")

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    # Save the downloaded file locally
    cache_path.write_bytes(response.content)

    print(f"Saved Census Gazetteer: {cache_path.name}")

    return cache_path

In [12]:
def collect_county_climate(county, start_year, end_year):
    """
    Retrieve, validate, and aggregate NASA POWER climate data
    for a single county.

    Parameters
    ----------
    county : pd.Series
        County record containing FIPS, latitude, and longitude.
    start_year : int
        First year of climate observations.
    end_year : int
        Last year of climate observations.

    Returns
    -------
    pd.DataFrame
        Validated annual climate statistics with county identifiers.
    """

    # Retrieve daily NASA POWER observations
    data = fetch_nasa_climate(
        latitude=county["LATITUDE"],
        longitude=county["LONGITUDE"],
        start_year=start_year,
        end_year=end_year
    )

    # Convert API response into a cleaned DataFrame
    daily_df = process_nasa_climate(data)

    # Validate daily observations before aggregation
    validate_daily_climate(
        daily_df=daily_df,
        start_year=start_year,
        end_year=end_year
    )

    # Aggregate validated daily observations
    annual_df = aggregate_annual_climate(daily_df)

    # Aggregate growing-season climate observations (May–September)
    seasonal_df = aggregate_growing_season_climate(
        climate_df=daily_df,
        start_month=5,
        end_month=9
    )

    # Combine annual and growing-season statistics by year
    annual_df = annual_df.merge(
        seasonal_df,
        on="YEAR",
        how="left",
        validate="one_to_one"
    )

    # Verify the expected number of annual observations
    expected_years = end_year - start_year + 1

    if len(annual_df) != expected_years:
        raise ValueError(
            f"Expected {expected_years} annual records, "
            f"but received {len(annual_df)}."
        )

    # Attach geographic identifiers
    annual_df["FIPS"] = county["FIPS"]
    annual_df["STATE_NAME"] = county["STATE_NAME"]
    annual_df["COUNTY_NAME"] = county["COUNTY_NAME"]

    return annual_df

In [13]:
def collect_all_county_climate(
    counties,
    start_year,
    end_year,
    checkpoint_path,
    request_delay=1.0,
    checkpoint_interval=10
):
    """
    Collect annual NASA POWER climate data for multiple counties.

    Supports:
    - Resuming from an existing checkpoint
    - Skipping previously completed counties
    - Validating each county's climate observations
    - Saving progress incrementally
    - Retrying failed API requests through the existing helper
    """

    expected_years = set(range(start_year, end_year + 1))

    # Load previously collected records if a checkpoint exists
    if checkpoint_path.exists():
        results_df = pd.read_csv(
            checkpoint_path,
            dtype={"FIPS": str}
        )

        print(
            f"Loaded checkpoint: {len(results_df):,} annual records."
        )
    else:
        results_df = pd.DataFrame()

    # Identify counties with complete annual and seasonal climate data
    required_columns = [
        "AVG_TEMPERATURE_C",
        "TOTAL_PRECIPITATION_MM",
        "GROWING_SEASON_AVG_TEMPERATURE_C",
        "GROWING_SEASON_TOTAL_PRECIPITATION_MM"
    ]

    completed_counties = set()

    if not results_df.empty:
        has_required_columns = all(
            column in results_df.columns
            for column in required_columns
        )

        if has_required_columns:
            for fips, group in results_df.groupby("FIPS"):
                observed_years = set(group["YEAR"])

                if (
                    observed_years == expected_years
                    and len(group) == len(expected_years)
                    and group[required_columns].notna().all().all()
                ):
                    completed_counties.add(fips)

    print(f"Completed counties: {len(completed_counties)}")
    print(f"Remaining counties: {len(counties) - len(completed_counties)}")

    new_results = []
    failures = []
    processed_since_checkpoint = 0

    for _, county in counties.iterrows():
        fips = county["FIPS"]

        if fips in completed_counties:
            continue

        print(
            f"\nCollecting {county['COUNTY_NAME']}, "
            f"{county['STATE_NAME']} (FIPS: {fips})"
        )

        try:
            county_df = collect_county_climate(
                county=county,
                start_year=start_year,
                end_year=end_year
            )

            # Replace any incomplete records for this county
            if not results_df.empty:
                results_df = results_df[
                    results_df["FIPS"] != fips
                ]

            new_results.append(county_df)
            processed_since_checkpoint += 1

            # Save progress at regular intervals
            if processed_since_checkpoint >= checkpoint_interval:
                results_df = pd.concat(
                    [results_df, *new_results],
                    ignore_index=True
                )

                results_df.to_csv(checkpoint_path, index=False)

                print(
                    f"\nCheckpoint saved: "
                    f"{len(results_df):,} annual records."
                )

                new_results = []
                processed_since_checkpoint = 0

            # Pause between county requests
            time.sleep(request_delay)

        except Exception as error:
            print(f"Failed: {fips} — {error}")
            failures.append({
                "FIPS": fips,
                "COUNTY_NAME": county["COUNTY_NAME"],
                "STATE_NAME": county["STATE_NAME"],
                "ERROR": str(error)
            })

    # Save any remaining successful results
    if new_results:
        results_df = pd.concat(
            [results_df, *new_results],
            ignore_index=True
        )

    if not results_df.empty:
        results_df = results_df.sort_values(
            ["FIPS", "YEAR"]
        ).reset_index(drop=True)

        results_df.to_csv(checkpoint_path, index=False)

    print("\nCollection complete.")
    print("Annual records:", len(results_df))
    print("Counties collected:", results_df["FIPS"].nunique() if not results_df.empty else 0)
    print("Failed counties:", len(failures))

    return results_df, pd.DataFrame(failures)

## NASA POWER API Configuration

This section defines the configuration settings used to retrieve historical climate data from the NASA POWER API.

The configuration includes the API endpoint, climate parameters, study period (2015–2025), and geographic coordinates for our initial test location near Des Moines, Iowa.

Centralizing these settings allows us to reuse the same configuration throughout the notebook, making it easier to expand data collection to additional counties and maintain consistency with our USDA dataset.

In [21]:
# NASA POWER API configuration
NASA_API_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"

NASA_CONFIG = {
    "parameters": ["T2M", "PRECTOTCORR"],
    "community": "AG",
    "start_year": 2015,
    "end_year": 2025,
    "format": "JSON",
}

# Initial test location: near Des Moines, Iowa
TEST_LOCATION = {
    "latitude": 41.59,
    "longitude": -93.62,
}

# NASA POWER cache directory
NASA_CACHE_DIR = DATA_DIR / "cache" / "nasa_power"
NASA_CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [25]:
# Test NASA POWER data retrieval for Des Moines, Iowa
data = fetch_nasa_climate(
    latitude=TEST_LOCATION["latitude"],
    longitude=TEST_LOCATION["longitude"],
    start_year=NASA_CONFIG["start_year"],
    end_year=NASA_CONFIG["start_year"],
)

# Inspect the returned climate parameters
print("Available parameters:")
print(list(data["properties"]["parameter"].keys()))

print("\nSample temperature observations:")
print(
    list(data["properties"]["parameter"]["T2M"].items())[:5]
)

Loading cached NASA POWER data: nasa_power_d1ba09fe58f2acce.json
Available parameters:
['T2M', 'PRECTOTCORR']

Sample temperature observations:
[('20150101', -6.57), ('20150102', -4.1), ('20150103', -2.35), ('20150104', -13.32), ('20150105', -14.27)]


### Processing NASA POWER API Responses

The initial API request successfully retrieved daily temperature and precipitation measurements for our test location near Des Moines, Iowa.

NASA POWER returns data in JSON format, with each environmental variable stored separately and indexed by date. To facilitate exploratory analysis, we will convert these observations into a Pandas DataFrame containing one row per day.

We will also convert the date strings into datetime objects and examine the resulting dataset for missing values before performing any temporal aggregation.

In [27]:
# Process daily climate observations
climate_df = process_nasa_climate(data)

display(climate_df.head())

print("\nDataset shape:", climate_df.shape)

print("\nMissing values:")
print(climate_df.isna().sum())

,DATE,TEMPERATURE_C,PRECIPITATION_MM
0,2015-01-01,-6.57,0.05
1,2015-01-02,-4.10,0.06
2,2015-01-03,-2.35,0.63
3,2015-01-04,-13.32,0.86
4,2015-01-05,-14.27,5.92



Dataset shape: (365, 3)

Missing values:
DATE                0
TEMPERATURE_C       0
PRECIPITATION_MM    0
dtype: int64


In [28]:
# Validate daily climate data coverage
expected_dates = pd.date_range(
    start="2015-01-01",
    end="2015-12-31",
    freq="D"
)

actual_dates = pd.DatetimeIndex(climate_df["DATE"])

missing_dates = expected_dates.difference(actual_dates)
duplicate_dates = climate_df["DATE"].duplicated().sum()

print(f"Expected days: {len(expected_dates)}")
print(f"Observed days: {len(climate_df)}")
print(f"Missing dates: {len(missing_dates)}")
print(f"Duplicate dates: {duplicate_dates}")

Expected days: 365
Observed days: 365
Missing dates: 0
Duplicate dates: 0


### Climate Data Validation

The NASA POWER API returned 365 daily observations for our test location in 2015, containing average air temperature in degrees Celsius and precipitation in millimeters per day.

During processing, NASA POWER missing-value placeholders, such as `-999`, were converted into missing values (`NaN`) to prevent them from distorting subsequent calculations.

Initial validation confirmed that the dataset contains all 365 expected dates, with no duplicate dates or missing temperature or precipitation measurements.

As an additional validation step, we will examine the minimum and maximum values of each climate variable to identify potentially unusual observations before calculating annual and growing-season summaries.

In [29]:
# Check for missing measurements after cleaning
print("Missing values:")
print(climate_df.isna().sum())

# Examine measurement ranges and descriptive statistics
print("\nClimate measurement summary:")

display(
    climate_df[
        ["TEMPERATURE_C", "PRECIPITATION_MM"]
    ].describe().round(2)
)

Missing values:
DATE                0
TEMPERATURE_C       0
PRECIPITATION_MM    0
dtype: int64

Climate measurement summary:


,TEMPERATURE_C,PRECIPITATION_MM
count,365.00,365.00
mean,10.36,3.08
std,11.16,7.41
min,-19.06,0.00
25%,2.06,0.00
50%,12.00,0.06
75%,20.29,1.76
max,27.56,59.48


### Aggregating Daily Climate Data

The NASA POWER dataset provides daily temperature and precipitation measurements, while our USDA agricultural dataset reports crop yields annually.

To prepare these datasets for integration, we will aggregate daily climate observations into annual summaries.

Average annual temperature will be calculated as the arithmetic mean of daily temperatures, while total annual precipitation will be calculated by summing daily precipitation measurements.

These aggregated variables provide an initial representation of environmental conditions for each year. However, annual measurements may not fully capture the conditions most relevant to crop development. We may subsequently calculate growing-season statistics to better investigate relationships between climate and agricultural productivity.

In [31]:
# Aggregate daily observations into annual climate statistics
annual_climate = aggregate_annual_climate(climate_df)

# Display rounded values for readability
display(annual_climate.round(2))

,YEAR,AVG_TEMPERATURE_C,TOTAL_PRECIPITATION_MM
0,2015,10.36,1123.19


### Preparing Geographic Locations for Climate Data Collection

After successfully retrieving and aggregating NASA POWER measurements for a single test location, the next objective is to expand climate data collection to all counties represented in our agricultural dataset.

NASA POWER provides point-based environmental measurements using geographic coordinates. Our USDA dataset, however, identifies observations by state and county.

To connect these sources, we will construct a geographic reference table containing each unique county and its corresponding coordinates. We will initially use county centroid locations to approximate local climate conditions, recognizing that a single point may not capture all environmental variation within a county.

We will begin by identifying the unique counties in our cleaned agricultural dataset.

In [33]:
# Identify unique counties in our cleaned dataset
unique_counties = (
    clean_df[
        ["STATE_NAME", "STATE_ANSI", "COUNTY_NAME", "COUNTY_ANSI"]
    ]
    .drop_duplicates()
    .sort_values(["STATE_NAME", "COUNTY_NAME"])
    .reset_index(drop=True)
)

print(f"Total unique counties: {len(unique_counties)}")

print("\nCounties by state:")
print(unique_counties["STATE_NAME"].value_counts())

display(unique_counties.head(10))

Total unique counties: 203

Counties by state:
STATE_NAME
ILLINOIS    103
IOWA        100
Name: count, dtype: int64


,STATE_NAME,STATE_ANSI,COUNTY_NAME,COUNTY_ANSI
0,ILLINOIS,17,ADAMS,1.0
1,ILLINOIS,17,ALEXANDER,3.0
2,ILLINOIS,17,BOND,5.0
3,ILLINOIS,17,BOONE,7.0
4,ILLINOIS,17,BROWN,9.0
5,ILLINOIS,17,BUREAU,11.0
6,ILLINOIS,17,CALHOUN,13.0
7,ILLINOIS,17,CARROLL,15.0
8,ILLINOIS,17,CASS,17.0
9,ILLINOIS,17,CHAMPAIGN,19.0


#### Identifying Valid Geographic Counties

Before retrieving geographic coordinates, we need to identify the unique counties represented in our USDA agricultural dataset.

Our initial inspection identified 203 county entries across Illinois and Iowa. However, two entries were labeled **OTHER COUNTIES**, representing aggregated USDA reporting categories rather than individual counties. These records also lacked county ANSI identifiers, making them unsuitable for geographic matching.

To construct an accurate geographic reference table, we exclude entries with missing county ANSI identifiers, remove duplicate county records, and sort the remaining counties by state and name.

This produces **201 unique counties: 102 in Illinois and 99 in Iowa**.

The original agricultural dataset remains unchanged. This filtering applies only to the geographic reference table used to retrieve county coordinates and connect USDA crop yields with NASA POWER climate observations.|

In [35]:
# Identify unique geographic counties in our cleaned dataset
unique_counties = (
    clean_df[
        ["STATE_NAME", "STATE_ANSI", "COUNTY_NAME", "COUNTY_ANSI"]
    ]
    .dropna(subset=["COUNTY_ANSI"])
    .drop_duplicates()
    .sort_values(["STATE_NAME", "COUNTY_NAME"])
    .reset_index(drop=True)
)

print(f"Total unique counties: {len(unique_counties)}")

print("\nCounties by state:")
print(unique_counties["STATE_NAME"].value_counts())

display(unique_counties.head(10))

Total unique counties: 201

Counties by state:
STATE_NAME
ILLINOIS    102
IOWA         99
Name: count, dtype: int64


,STATE_NAME,STATE_ANSI,COUNTY_NAME,COUNTY_ANSI
0,ILLINOIS,17,ADAMS,1.0
1,ILLINOIS,17,ALEXANDER,3.0
2,ILLINOIS,17,BOND,5.0
3,ILLINOIS,17,BOONE,7.0
4,ILLINOIS,17,BROWN,9.0
5,ILLINOIS,17,BUREAU,11.0
6,ILLINOIS,17,CALHOUN,13.0
7,ILLINOIS,17,CARROLL,15.0
8,ILLINOIS,17,CASS,17.0
9,ILLINOIS,17,CHAMPAIGN,19.0


### Standardizing County FIPS Codes

To associate USDA agricultural records with geographic coordinates, we will construct standardized five-digit county FIPS codes using the state and county ANSI identifiers.

Each FIPS code consists of a two-digit state identifier followed by a three-digit county identifier. Leading zeros must be preserved to ensure compatibility with geographic reference datasets.

These identifiers will provide a consistent method for matching counties across data sources, avoiding potential inconsistencies in county names or formatting.

We will retain the original state and county identifiers alongside the standardized FIPS codes for transparency and future data integration.

In [37]:
# Construct standardized five-digit county FIPS codes
unique_counties["FIPS"] = (
    unique_counties["STATE_ANSI"].astype(int).astype(str).str.zfill(2)
    +
    unique_counties["COUNTY_ANSI"].astype(int).astype(str).str.zfill(3)
)

# Validate FIPS code formatting and uniqueness
print("Total counties:", len(unique_counties))
print("Unique FIPS codes:", unique_counties["FIPS"].nunique())
print("Invalid FIPS lengths:", (unique_counties["FIPS"].str.len() != 5).sum())

display(unique_counties.head(10))

Total counties: 201
Unique FIPS codes: 201
Invalid FIPS lengths: 0


,STATE_NAME,STATE_ANSI,COUNTY_NAME,COUNTY_ANSI,FIPS
0,ILLINOIS,17,ADAMS,1.0,17001
1,ILLINOIS,17,ALEXANDER,3.0,17003
2,ILLINOIS,17,BOND,5.0,17005
3,ILLINOIS,17,BOONE,7.0,17007
4,ILLINOIS,17,BROWN,9.0,17009
5,ILLINOIS,17,BUREAU,11.0,17011
6,ILLINOIS,17,CALHOUN,13.0,17013
7,ILLINOIS,17,CARROLL,15.0,17015
8,ILLINOIS,17,CASS,17.0,17017
9,ILLINOIS,17,CHAMPAIGN,19.0,17019


### Retrieving County Geographic Coordinates

NASA POWER retrieves climate observations using geographic coordinates, while our USDA dataset identifies agricultural observations by county FIPS code.

To connect these datasets, we will use the U.S. Census Bureau's county Gazetteer reference file, which provides standardized geographic identifiers and representative latitude and longitude coordinates for U.S. counties.

We will match the Census geographic identifiers to our 201 county FIPS codes, retaining only counties included in the agricultural dataset.

The Census coordinates represent internal points within county boundaries rather than county-wide geographic averages. We will use these points to approximate local climate conditions, recognizing that environmental conditions may vary across each county.

The geographic reference file will be cached locally to support reproducibility and avoid unnecessary repeated downloads.

In [41]:
# U.S. Census Bureau county geographic reference
CENSUS_GAZETTEER_URL = (
    "https://www2.census.gov/geo/docs/maps-data/data/"
    "gazetteer/2025_Gazetteer/2025_Gaz_counties_national.zip"
)

# Local geographic reference directory
GEO_DIR = DATA_DIR / "geographic"
GEO_DIR.mkdir(parents=True, exist_ok=True)

# Cached Census Gazetteer file
CENSUS_CACHE_PATH = GEO_DIR / "2025_Gazetteer_counties_national.zip"

print("Geographic reference directory:", GEO_DIR)
print("Census cache path:", CENSUS_CACHE_PATH)

Geographic reference directory: c:\Users\Jenny\OneDrive\ASU\DAT301\agricultural-climate-analysis\data\geographic
Census cache path: c:\Users\Jenny\OneDrive\ASU\DAT301\agricultural-climate-analysis\data\geographic\2025_Gazetteer_counties_national.zip


In [43]:
# Download or load the Census county geographic reference
census_file = download_census_gazetteer(
    url=CENSUS_GAZETTEER_URL,
    cache_path=CENSUS_CACHE_PATH
)

print("\nFile location:", census_file)
print("File size:", f"{census_file.stat().st_size / 1024:.1f} KB")

Loading cached Census Gazetteer: 2025_Gazetteer_counties_national.zip

File location: c:\Users\Jenny\OneDrive\ASU\DAT301\agricultural-climate-analysis\data\geographic\2025_Gazetteer_counties_national.zip
File size: 135.7 KB


### Inspecting Census County Geographic Data

The Census Gazetteer reference file has been downloaded and cached locally.

We will now load the geographic reference data into Pandas and examine its structure, including the available county identifiers and geographic coordinates.

Before merging this reference with our agricultural dataset, we will verify that the necessary fields are present and determine how the Census identifiers correspond to our standardized five-digit county FIPS codes.

In [45]:
# Load the cached Census Gazetteer file
census_df = pd.read_csv(
    census_file,
    sep="|",
    dtype={"GEOID": str}
)

# Remove whitespace from column names
census_df.columns = census_df.columns.str.strip()

print("Dataset shape:", census_df.shape)

print("\nAvailable columns:")
print(census_df.columns.tolist())

print("\nSample records:")
display(census_df.head())

Dataset shape: (3222, 11)

Available columns:
['USPS', 'GEOID', 'GEOIDFQ', 'ANSICODE', 'NAME', 'ALAND', 'AWATER', 'ALAND_SQMI', 'AWATER_SQMI', 'INTPTLAT', 'INTPTLONG']

Sample records:


,USPS,GEOID,GEOIDFQ,ANSICODE,NAME,ALAND,AWATER,ALAND_SQMI,AWATER_SQMI,INTPTLAT,INTPTLONG
0,AL,01001,0500000US01001,161526,Autauga County,1539631460,25677536,594.455,9.914,32.532237,-86.646440
1,AL,01003,0500000US01003,161527,Baldwin County,4117933903,1132678359,1589.943,437.330,30.659218,-87.746067
2,AL,01005,0500000US01005,161528,Barbour County,2292160152,50523213,885.008,19.507,31.870253,-85.405103
3,AL,01007,0500000US01007,161529,Bibb County,1612188713,9572302,622.470,3.696,33.015893,-87.127148
4,AL,01009,0500000US01009,161530,Blount County,1670296790,14822589,644.905,5.723,33.977358,-86.566440


### Matching USDA Counties with Census Coordinates

The Census Gazetteer dataset provides standardized county FIPS identifiers and representative geographic coordinates.

We will merge these coordinates with our USDA county reference table using the five-digit FIPS codes constructed earlier.

A left join will preserve all 201 counties in our agricultural dataset, allowing us to identify any counties without matching Census geographic information.

We will validate the results by checking the number of matched counties, missing coordinates, and duplicate identifiers before proceeding with NASA POWER data collection.

In [46]:
# Select the geographic fields needed for climate collection
county_coordinates = census_df[
    ["GEOID", "INTPTLAT", "INTPTLONG"]
].copy()

# Rename columns for consistency
county_coordinates = county_coordinates.rename(columns={
    "GEOID": "FIPS",
    "INTPTLAT": "LATITUDE",
    "INTPTLONG": "LONGITUDE"
})

# Merge USDA county identifiers with Census coordinates
county_locations = unique_counties.merge(
    county_coordinates,
    on="FIPS",
    how="left",
    validate="one_to_one"
)

# Validate geographic matching
print("Total counties:", len(county_locations))
print("Counties with coordinates:",
      county_locations["LATITUDE"].notna().sum())

print("Missing coordinates:")
print(county_locations[["LATITUDE", "LONGITUDE"]].isna().sum())

print("Duplicate FIPS codes:",
      county_locations["FIPS"].duplicated().sum())

display(county_locations.head(10))

Total counties: 201
Counties with coordinates: 201
Missing coordinates:
LATITUDE     0
LONGITUDE    0
dtype: int64
Duplicate FIPS codes: 0


,STATE_NAME,STATE_ANSI,COUNTY_NAME,COUNTY_ANSI,FIPS,LATITUDE,LONGITUDE
0,ILLINOIS,17,ADAMS,1.0,17001,39.986052,-91.194961
1,ILLINOIS,17,ALEXANDER,3.0,17003,37.183658,-89.349516
2,ILLINOIS,17,BOND,5.0,17005,38.885924,-89.436592
3,ILLINOIS,17,BOONE,7.0,17007,42.318983,-88.824295
4,ILLINOIS,17,BROWN,9.0,17009,39.962069,-90.750309
5,ILLINOIS,17,BUREAU,11.0,17011,41.401304,-89.528377
6,ILLINOIS,17,CALHOUN,13.0,17013,39.164262,-90.666295
7,ILLINOIS,17,CARROLL,15.0,17015,42.070900,-89.924190
8,ILLINOIS,17,CASS,17.0,17017,39.969202,-90.245704
9,ILLINOIS,17,CHAMPAIGN,19.0,17019,40.138985,-88.196975


### Exporting the County Geographic Reference

After matching USDA county identifiers with the Census Gazetteer data, we obtained geographic coordinates for all 201 counties in our study.

We will export this reference table to a CSV file in the project's processed data directory. This allows subsequent notebooks to reuse the county identifiers and coordinates without repeating the geographic matching process.

The exported dataset includes state and county names, ANSI identifiers, standardized five-digit FIPS codes, and representative latitude and longitude coordinates.

This reference will serve as the geographic foundation for collecting county-level NASA POWER climate observations.

In [47]:
# Define the output path
COUNTY_LOCATIONS_PATH = (
    PROCESSED_DIR / "County_Geographic_Reference.csv"
)

# Ensure the processed directory exists
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Export the geographic reference table
county_locations.to_csv(
    COUNTY_LOCATIONS_PATH,
    index=False
)

print(f"Exported {len(county_locations)} county records.")
print(f"Saved to: {COUNTY_LOCATIONS_PATH}")

# Verify the exported file
verified_locations = pd.read_csv(
    COUNTY_LOCATIONS_PATH,
    dtype={"FIPS": str}
)

print(f"\nVerified records: {len(verified_locations)}")
print(f"Unique FIPS codes: {verified_locations['FIPS'].nunique()}")

Exported 201 county records.
Saved to: c:\Users\Jenny\OneDrive\ASU\DAT301\agricultural-climate-analysis\data\processed\County_Geographic_Reference.csv

Verified records: 201
Unique FIPS codes: 201


### Testing Multi-County Climate Data Collection

With geographic coordinates available for all 201 counties, we can expand NASA POWER data collection beyond our initial test location.

We will first retrieve climate observations for a small sample of three counties to validate the collection process before requesting the complete dataset.

For each county, we will use its representative latitude and longitude to retrieve daily temperature and precipitation measurements for 2015 through 2025.

Our existing caching mechanism will store API responses locally, allowing interrupted collection runs to resume without downloading previously retrieved observations.

Once the sample collection is validated, we will extend the process to all counties.

In [48]:
# Select three counties for an initial collection test
test_counties = (
    county_locations
    .groupby("STATE_NAME", group_keys=False)
    .head(2)
    .head(3)
    .copy()
)

display(
    test_counties[
        ["STATE_NAME", "COUNTY_NAME", "FIPS", "LATITUDE", "LONGITUDE"]
    ]
)

,STATE_NAME,COUNTY_NAME,FIPS,LATITUDE,LONGITUDE
0,ILLINOIS,ADAMS,17001,39.986052,-91.194961
1,ILLINOIS,ALEXANDER,17003,37.183658,-89.349516
102,IOWA,ADAIR,19001,41.328528,-94.478164


### Sanity Checks: County Climate Collection

Before collecting climate data for all 201 counties, we will test our reusable collection and validation functions using Adams County, Illinois.

These checks verify that:

- The collection function retrieves and aggregates 11 years of climate observations.
- The dataset contains all expected daily records, including leap years.
- No dates are missing or duplicated.
- Temperature and precipitation measurements contain no missing values.
- Cached API responses can be reused without additional downloads.

These tests help identify potential problems before expanding climate data collection to the full study area.

In [83]:
# Test climate collection for Adams County, Illinois
test_county = test_counties.iloc[0]

adams_climate = collect_county_climate(
    county=test_county,
    start_year=NASA_CONFIG["start_year"],
    end_year=NASA_CONFIG["end_year"]
)

print("Annual observations:", len(adams_climate))

display(adams_climate.head())

Loading cached NASA POWER data: nasa_power_caa46587dc714077.json
Validation passed: 4,018 daily observations (2015–2025).
Annual observations: 11


,YEAR,AVG_TEMPERATURE_C,TOTAL_PRECIPITATION_MM,GROWING_SEASON_AVG_TEMPERATURE_C,GROWING_SEASON_TOTAL_PRECIPITATION_MM,FIPS,STATE_NAME,COUNTY_NAME
0,2015,11.939041,1178.05,21.808889,701.02,17001,ILLINOIS,ADAMS
1,2016,12.730164,767.59,22.593856,504.84,17001,ILLINOIS,ADAMS
2,2017,12.995671,834.38,22.282941,387.88,17001,ILLINOIS,ADAMS
3,2018,11.883562,1002.00,24.399216,490.47,17001,ILLINOIS,ADAMS
4,2019,11.115233,1160.86,22.084183,658.43,17001,ILLINOIS,ADAMS


In [84]:
# Test growing-season aggregation for Adams County
adams_daily = process_nasa_climate(
    fetch_nasa_climate(
        latitude=test_county["LATITUDE"],
        longitude=test_county["LONGITUDE"],
        start_year=NASA_CONFIG["start_year"],
        end_year=NASA_CONFIG["end_year"]
    )
)

adams_growing_season = aggregate_growing_season_climate(
    climate_df=adams_daily,
    start_month=5,
    end_month=9
)

print("Growing-season annual records:", len(adams_growing_season))
print("Years covered:", sorted(adams_growing_season["YEAR"].tolist()))

display(adams_growing_season.head())

Loading cached NASA POWER data: nasa_power_caa46587dc714077.json
Growing-season annual records: 11
Years covered: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


,YEAR,GROWING_SEASON_AVG_TEMPERATURE_C,GROWING_SEASON_TOTAL_PRECIPITATION_MM
0,2015,21.808889,701.02
1,2016,22.593856,504.84
2,2017,22.282941,387.88
3,2018,24.399216,490.47
4,2019,22.084183,658.43


In [85]:
# Test climate collection across three counties
test_results = []

for _, county in test_counties.iterrows():
    print(
        f"\nProcessing {county['COUNTY_NAME']}, "
        f"{county['STATE_NAME']} (FIPS: {county['FIPS']})"
    )

    county_climate = collect_county_climate(
        county=county,
        start_year=NASA_CONFIG["start_year"],
        end_year=NASA_CONFIG["end_year"]
    )

    test_results.append(county_climate)

# Combine results into one DataFrame
test_climate_df = pd.concat(
    test_results,
    ignore_index=True
)

print("\nTotal annual observations:", len(test_climate_df))
print("Unique counties:", test_climate_df["FIPS"].nunique())
print("Years covered:", sorted(test_climate_df["YEAR"].unique()))

display(test_climate_df.head(10))


Processing ADAMS, ILLINOIS (FIPS: 17001)
Loading cached NASA POWER data: nasa_power_caa46587dc714077.json
Validation passed: 4,018 daily observations (2015–2025).

Processing ALEXANDER, ILLINOIS (FIPS: 17003)
Loading cached NASA POWER data: nasa_power_7128d177ebcbf371.json
Validation passed: 4,018 daily observations (2015–2025).

Processing ADAIR, IOWA (FIPS: 19001)
Loading cached NASA POWER data: nasa_power_7eef061410ab7ebb.json
Validation passed: 4,018 daily observations (2015–2025).

Total annual observations: 33
Unique counties: 3
Years covered: [np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]


,YEAR,AVG_TEMPERATURE_C,TOTAL_PRECIPITATION_MM,GROWING_SEASON_AVG_TEMPERATURE_C,GROWING_SEASON_TOTAL_PRECIPITATION_MM,FIPS,STATE_NAME,COUNTY_NAME
0,2015,11.939041,1178.05,21.808889,701.02,17001,ILLINOIS,ADAMS
1,2016,12.730164,767.59,22.593856,504.84,17001,ILLINOIS,ADAMS
2,2017,12.995671,834.38,22.282941,387.88,17001,ILLINOIS,ADAMS
3,2018,11.883562,1002.00,24.399216,490.47,17001,ILLINOIS,ADAMS
4,2019,11.115233,1160.86,22.084183,658.43,17001,ILLINOIS,ADAMS
5,2020,11.707541,924.89,21.375163,504.04,17001,ILLINOIS,ADAMS
6,2021,11.999945,1109.25,21.870000,571.94,17001,ILLINOIS,ADAMS
7,2022,11.598411,848.25,22.972092,437.22,17001,ILLINOIS,ADAMS
8,2023,13.384082,703.46,23.377516,321.60,17001,ILLINOIS,ADAMS
9,2024,13.421011,926.10,23.146405,411.31,17001,ILLINOIS,ADAMS


In [75]:
# Test checkpointing and resuming with three counties
TEST_CHECKPOINT_PATH = (
    DATA_DIR / "cache" / "nasa_power" / "test_collection_checkpoint.csv"
)

# Start with a clean test checkpoint
if TEST_CHECKPOINT_PATH.exists():
    TEST_CHECKPOINT_PATH.unlink()

test_checkpoint_df, test_failures = collect_all_county_climate(
    counties=test_counties,
    start_year=NASA_CONFIG["start_year"],
    end_year=NASA_CONFIG["end_year"],
    checkpoint_path=TEST_CHECKPOINT_PATH,
    request_delay=0,
    checkpoint_interval=2
)

print("\nCheckpoint file exists:", TEST_CHECKPOINT_PATH.exists())
print("Annual records:", len(test_checkpoint_df))
print("Unique counties:", test_checkpoint_df["FIPS"].nunique())
print("Failures:", len(test_failures))

Completed counties: 0
Remaining counties: 3

Loading cached NASA POWER data: nasa_power_caa46587dc714077.json
Validation passed: 4,018 daily observations (2015–2025).

Loading cached NASA POWER data: nasa_power_7128d177ebcbf371.json
Validation passed: 4,018 daily observations (2015–2025).

Checkpoint saved: 22 annual records.

Loading cached NASA POWER data: nasa_power_7eef061410ab7ebb.json
Validation passed: 4,018 daily observations (2015–2025).

Collection complete.
Annual records: 33
Counties collected: 3
Failed counties: 0

Checkpoint file exists: True
Annual records: 33
Unique counties: 3
Failures: 0


In [76]:
# Test resuming collection from an existing checkpoint
resumed_df, resumed_failures = collect_all_county_climate(
    counties=test_counties,
    start_year=NASA_CONFIG["start_year"],
    end_year=NASA_CONFIG["end_year"],
    checkpoint_path=TEST_CHECKPOINT_PATH,
    request_delay=0,
    checkpoint_interval=2
)

# Verify completed counties were skipped
assert len(resumed_df) == 33
assert resumed_df["FIPS"].nunique() == 3
assert len(resumed_failures) == 0

print("\nCheckpoint resume test passed.")

Loaded checkpoint: 33 annual records.
Completed counties: 3
Remaining counties: 0

Collection complete.
Annual records: 33
Counties collected: 3
Failed counties: 0

Checkpoint resume test passed.


In [87]:
# Test upgrading an older annual-only checkpoint

UPGRADE_TEST_PATH = (
    DATA_DIR / "cache" / "nasa_power" / "test_checkpoint_upgrade.csv"
)

# Simulate the older checkpoint format
annual_columns = [
    "YEAR",
    "AVG_TEMPERATURE_C",
    "TOTAL_PRECIPITATION_MM",
    "FIPS",
    "STATE_NAME",
    "COUNTY_NAME"
]

test_climate_df[annual_columns].to_csv(
    UPGRADE_TEST_PATH,
    index=False
)

# Recollect counties using the updated feature requirements
upgraded_df, upgrade_failures = collect_all_county_climate(
    counties=test_counties,
    start_year=NASA_CONFIG["start_year"],
    end_year=NASA_CONFIG["end_year"],
    checkpoint_path=UPGRADE_TEST_PATH,
    request_delay=0,
    checkpoint_interval=2
)

# Verify the upgraded dataset
assert len(upgraded_df) == 33
assert upgraded_df["FIPS"].nunique() == 3
assert upgraded_df.duplicated(["FIPS", "YEAR"]).sum() == 0
assert len(upgrade_failures) == 0

assert upgraded_df[
    [
        "GROWING_SEASON_AVG_TEMPERATURE_C",
        "GROWING_SEASON_TOTAL_PRECIPITATION_MM"
    ]
].notna().all().all()

print("\nCheckpoint upgrade test passed!")

Loaded checkpoint: 33 annual records.
Completed counties: 0
Remaining counties: 3

Loading cached NASA POWER data: nasa_power_caa46587dc714077.json
Validation passed: 4,018 daily observations (2015–2025).

Loading cached NASA POWER data: nasa_power_7128d177ebcbf371.json
Validation passed: 4,018 daily observations (2015–2025).

Checkpoint saved: 33 annual records.

Loading cached NASA POWER data: nasa_power_7eef061410ab7ebb.json
Validation passed: 4,018 daily observations (2015–2025).

Collection complete.
Annual records: 33
Counties collected: 3
Failed counties: 0

Checkpoint upgrade test passed!


In [68]:
# Test retry behavior using a simulated temporary server error
from unittest.mock import patch, Mock

failed_response = Mock()
failed_response.raise_for_status.side_effect = (
    requests.exceptions.HTTPError("503 Service Unavailable")
)

successful_response = Mock()
successful_response.raise_for_status.return_value = None

with patch(
    "requests.get",
    side_effect=[failed_response, successful_response]
) as mock_get, patch("time.sleep") as mock_sleep:

    response = request_with_retry(
        url=NASA_API_URL,
        params={"test": "retry"},
        max_attempts=3,
        delay=2
    )

    assert response is successful_response
    assert mock_get.call_count == 2
    mock_sleep.assert_called_once_with(2)

print("Retry test passed: request succeeded on the second attempt.")

Request failed (attempt 1/3): 503 Service Unavailable
Retrying in 2 seconds...
Retry test passed: request succeeded on the second attempt.


## Full County Climate Data Collection

This section collects NASA POWER climate observations for all 201 counties in the study area.

The collection process uses local caching, automatic validation, retry logic, and request pacing to support reliable data retrieval. Successfully collected county records will be saved incrementally so the process can resume if interrupted.

In [69]:
# Configuration for full county climate collection
COLLECTION_CONFIG = {
    "request_delay": 1.0,
    "checkpoint_interval": 10,
}

CLIMATE_CHECKPOINT_PATH = (
    PROCESSED_DIR / "NASA_County_Climate_Checkpoint.csv"
)

CLIMATE_OUTPUT_PATH = (
    PROCESSED_DIR / "NASA_County_Climate_Annual.csv"
)

print("Counties to collect:", len(county_locations))
print("Years per county:", NASA_CONFIG["end_year"] - NASA_CONFIG["start_year"] + 1)
print("Expected annual observations:",
      len(county_locations) *
      (NASA_CONFIG["end_year"] - NASA_CONFIG["start_year"] + 1))

Counties to collect: 201
Years per county: 11
Expected annual observations: 2211


In [88]:
# Collect annual climate data for all study counties
county_climate_df, collection_failures = collect_all_county_climate(
    counties=county_locations,
    start_year=NASA_CONFIG["start_year"],
    end_year=NASA_CONFIG["end_year"],
    checkpoint_path=CLIMATE_CHECKPOINT_PATH,
    request_delay=COLLECTION_CONFIG["request_delay"],
    checkpoint_interval=COLLECTION_CONFIG["checkpoint_interval"]
)

print("\nCollection summary")
print("Annual observations:", len(county_climate_df))
print("Unique counties:", county_climate_df["FIPS"].nunique())
print("Failed counties:", len(collection_failures))

if not collection_failures.empty:
    display(collection_failures)

Loaded checkpoint: 2,211 annual records.
Completed counties: 0
Remaining counties: 201

Loading cached NASA POWER data: nasa_power_caa46587dc714077.json
Validation passed: 4,018 daily observations (2015–2025).

Loading cached NASA POWER data: nasa_power_7128d177ebcbf371.json
Validation passed: 4,018 daily observations (2015–2025).

Loading cached NASA POWER data: nasa_power_656e5f010292ab5f.json
Validation passed: 4,018 daily observations (2015–2025).

Loading cached NASA POWER data: nasa_power_3983e9bc8929d9b8.json
Validation passed: 4,018 daily observations (2015–2025).

Loading cached NASA POWER data: nasa_power_1d79639bcefa7046.json
Validation passed: 4,018 daily observations (2015–2025).

Loading cached NASA POWER data: nasa_power_fec49d62fb4f6759.json
Validation passed: 4,018 daily observations (2015–2025).

Loading cached NASA POWER data: nasa_power_1d3f35ba335039e8.json
Validation passed: 4,018 daily observations (2015–2025).

Loading cached NASA POWER data: nasa_power_9aef4f23

### Validating the Complete Climate Dataset

Before exporting the collected climate data, we verify that every county has observations for all 11 study years, that county-year combinations are unique, and that the aggregated climate measurements contain no missing values.

In [ ]:
# Validate the complete county-level annual climate dataset

expected_years = set(
    range(
        NASA_CONFIG["start_year"],
        NASA_CONFIG["end_year"] + 1
    )
)

expected_counties = set(county_locations["FIPS"])

# Check county coverage
observed_counties = set(county_climate_df["FIPS"])

missing_counties = expected_counties - observed_counties
unexpected_counties = observed_counties - expected_counties

# Check annual coverage for each county
incomplete_counties = []

for fips, group in county_climate_df.groupby("FIPS"):
    if set(group["YEAR"]) != expected_years:
        incomplete_counties.append(fips)

# Check duplicate county-year combinations
duplicate_records = county_climate_df.duplicated(
    subset=["FIPS", "YEAR"]
).sum()

# Check all annual and growing-season climate measurements
climate_columns = [
    "AVG_TEMPERATURE_C",
    "TOTAL_PRECIPITATION_MM",
    "GROWING_SEASON_AVG_TEMPERATURE_C",
    "GROWING_SEASON_TOTAL_PRECIPITATION_MM"
]

missing_measurements = county_climate_df[
    climate_columns
].isna().sum()

print("Missing counties:", len(missing_counties))
print("Unexpected counties:", len(unexpected_counties))
print("Counties with incomplete year coverage:", len(incomplete_counties))
print("Duplicate county-year records:", duplicate_records)

print("\nMissing climate measurements:")
print(missing_measurements)

assert len(missing_counties) == 0
assert len(unexpected_counties) == 0
assert len(incomplete_counties) == 0
assert duplicate_records == 0
assert missing_measurements.sum() == 0

print("\nFull climate dataset validation passed!")

Missing counties: 0
Unexpected counties: 0
Counties with incomplete year coverage: 0
Duplicate county-year records: 0

Missing climate measurements:
AVG_TEMPERATURE_C                        0
TOTAL_PRECIPITATION_MM                   0
GROWING_SEASON_AVG_TEMPERATURE_C         0
GROWING_SEASON_TOTAL_PRECIPITATION_MM    0
dtype: int64

Full climate dataset validation passed!


### Exporting the Validated Annual Climate Dataset

The validated NASA POWER climate observations are exported as a CSV file for use in subsequent notebooks.

Each record represents one county and year, with annual average temperature and total precipitation derived from daily NASA POWER observations.

County FIPS codes are preserved as five-character identifiers to support reliable merging with USDA crop yield data in Notebook 3.

In [90]:
# Export validated annual climate observations
county_climate_df.to_csv(
    CLIMATE_OUTPUT_PATH,
    index=False
)

# Verify the exported file
exported_climate_df = pd.read_csv(
    CLIMATE_OUTPUT_PATH,
    dtype={"FIPS": str}
)

assert len(exported_climate_df) == len(county_climate_df)
assert exported_climate_df["FIPS"].nunique() == 201
assert exported_climate_df.duplicated(["FIPS", "YEAR"]).sum() == 0

print(f"Exported {len(exported_climate_df):,} annual observations.")
print(f"Unique counties: {exported_climate_df['FIPS'].nunique()}")
print(f"Saved to: {CLIMATE_OUTPUT_PATH}")

Exported 2,211 annual observations.
Unique counties: 201
Saved to: c:\Users\Jenny\OneDrive\ASU\DAT301\agricultural-climate-analysis\data\processed\NASA_County_Climate_Annual.csv
